### Mounting Google Drive for Persistent Storage

To save files that persist across Colab sessions, you can mount your Google Drive. This will allow you to read from and write to your Drive directly from your Colab notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

UNIQUE_FILE = "/content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json" # Switched extension
SOURCE_FILE = "/content/drive/MyDrive/NLP_twitter/malaria_tweets.jsonl"

if os.path.exists(UNIQUE_FILE):
    print(f"--- {UNIQUE_FILE} exists. Loading unique data. ---")
    df_unique = pd.read_json(UNIQUE_FILE)
else:
    print("--- Cleaning and deduplicating... ---")
    df = pd.read_json(SOURCE_FILE, lines=True)

    # Selection and Renaming
    df = df[["text", "created_at", "author", "views", "favorite_count", "is_reply"]].copy()
    df.rename(columns={"created_at": "date", "author": "user", "favorite_count": "likes", }, inplace=True)

    # Cleaning
    df['text'] = df['text'].str.strip('"').str.strip()
    df_unique = df.drop_duplicates(subset=['text']).copy()
    df_unique["likes"] = pd.to_numeric(df_unique["likes"], errors='coerce').fillna(0)

    # SAVE AS JSON
    # orient='records' makes it a list of dictionaries: [{...}, {...}]
    # indent=4 makes it pretty-printed
    df_unique.to_json(UNIQUE_FILE, orient='records', indent=4, force_ascii=False)
    print(f"--- Saved {len(df_unique)} unique tweets to {UNIQUE_FILE} ---")

print(df_unique.head())

--- /content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json exists. Loading unique data. ---
                                                text                date  \
0  @konnectckris We get malaria. Man U get fever.... 2022-01-31 22:44:19   
1  @_Khicked Hard guy na only for women o, and ma... 2022-01-31 10:21:18   
2     @Mz_aheesha It is like you have slight malaria 2022-01-30 08:20:54   
3  No one persuades us to buy the malaria drug wh... 2022-01-29 09:29:37   
4  @Adebomi_Balogun @Xoxo_Vibez Is this ment or m... 2022-01-28 16:53:33   

              user  views  likes  is_reply  is_quote  label  \
0    MaziNnamdiOnu      0      0      True     False      5   
1      ___Motolani      0      1      True     False      5   
2  Femi_OfMainland      0      0      True     False      2   
3     Sir_Jattolee      0      3     False     False      2   
4  ChiedozieRapha2      0      0      True     False      5   

                  label_name annotator  
0                 Irrele

In [ ]:
!pip install -q transformers[torch] datasets accelerate scikit-learn

In [ ]:
from transformers import pipeline
import pandas as pd

# 1. Load your data
df = pd.read_json('/content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json')

# 2. Setup the classifier (using a powerful, lightweight model)
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=0) # Uses the Colab GPU

# 3. Define your methodology's categories
candidate_labels = [
    "Symptoms", "Treatment", "Prevention",
    "Misinformation", "Health System", "Personal Experience", "Irrelevant".

]

def classify_tweet(text):
    result = classifier(text, candidate_labels)
    # Return the label with the highest score
    return result['labels'][0]

from tqdm.auto import tqdm  # This adds a progress bar

# 4. Run the AI using Batch Processing
print("Starting fast batch classification...")

# Parameters for speed
batch_size = 16  # Adjust based on GPU memory (8, 16, or 32)
texts = df['text'].tolist()
results = []

# Process in batches instead of .apply()
for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i : i + batch_size]
    # Passing a list to the classifier enables parallel processing
    batch_results = classifier(batch_texts, candidate_labels)

    # Handle both single and multi-batch returns
    if isinstance(batch_results, dict): batch_results = [batch_results]

    # Get the top label for each tweet in the batch
    results.extend([res['labels'][0] for res in batch_results])
  # 5. Attach the results to your DataFrame
df['ai_label'] = results

# 6. Save the results to a NEW JSON file
# orient='records' makes it easy to read in other apps; indent=4 makes it human-readable
df.to_json('ai_labeled_tweets_final.json', orient='records', indent=4)

# 7. Print a preview to confirm
print("\nClassification Complete! File saved as 'ai_labeled_tweets_final.json'")
print(df[['text', 'ai_label']].head(10)) # Shows the first 10 tweets and their labels

# 8. (Optional) Download the file if you are in Google Colab
from google.colab import files
files.download('ai_labeled_tweets_final.json')


SyntaxError: invalid syntax (1915201481.py, line 17)

In [ ]:
# Create a mapping dictionary
label_map = {label: i for i, label in enumerate(candidate_labels)}
df['label'] = df['ai_label'].map(label_map)

# Split into Training (85%) and Testing (15%) as per your Methodology
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(df, test_size=0.15, random_state=42)

### Phase 1: Model Training (The "Engine") - Part 1: Dataset Prep and Preprocessing

First, we'll define the 5-class taxonomy as specified in the plan and map the existing 7-class labels to these new categories. Then, we will split the dataset into training, validation, and testing sets, and preprocess the text for BERTweet.

In [ ]:
import pandas as pd
import re
from datasets import Dataset
from sklearn.model_selection import train_test_split

# 1. Define the 5-class taxonomy
id2label = {1: "Symptoms & Burden", 2: "Treatment & Health System",
            3: "Prevention & Awareness", 4: "Misinformation", 5: "Irrelevant"}
label2id = {v: k for k, v in id2label.items()}

# Map the previous 7-class labels to the new 5-class taxonomy (1-5)
# Using the ai_label from the previous zero-shot classification
# Ensure df is loaded from 'unique_malaria_tweets.json'
FILE_PATH = '/content/drive/MyDrive/NLP_twitter/unique_malaria_tweets.json'
df = pd.read_json(FILE_PATH)

# Previous candidate_labels for reference (from cell zMM5b1-tXuxe)
# candidate_labels = ["Symptoms", "Treatment", "Prevention", "Misinformation", "Health System", "Personal Experience", "Irrelevant"]

# Define the mapping from old 7-class labels to new 5-class labels (1-5)
old_label_str_to_new_int = {
    "Symptoms": 1, # Symptoms & Burden
    "Treatment": 2, # Treatment & Health System
    "Prevention": 3, # Prevention & Awareness
    "Misinformation": 4, # Misinformation
    "Health System": 2, # Treatment & Health System
    "Personal Experience": 1, # Symptoms & Burden (assuming personal experience often relates to symptoms or direct impact)
    "Irrelevant": 5 # Irrelevant
}

# Apply the mapping to create the new 'label' column (1-5)
df['label'] = df['ai_label'].map(old_label_str_to_new_int)

# Ensure labels are integers
df['label'] = df['label'].astype(int)

print("New 5-class taxonomy defined and labels remapped:")
print(df[['ai_label', 'label']].head())
print("Value counts for new labels (1-5):")
print(df['label'].value_counts())


In [ ]:
# 2. Dataset Prep: Stratified 80/10/10 split
# Stratify on label to handle class imbalance

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

# Convert DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_dict(train_df[['text', 'label']])
val_dataset = Dataset.from_dict(val_df[['text', 'label']])
test_dataset = Dataset.from_dict(test_df[['text', 'label']])

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

print("\nClass distribution in Training set:")
print(train_df['label'].value_counts(normalize=True))
print("\nClass distribution in Validation set:")
print(val_df['label'].value_counts(normalize=True))
print("\nClass distribution in Test set:")
print(test_df['label'].value_counts(normalize=True))


In [ ]:
# 3. Preprocessing (BERTweet-specific)
# BERTweet expects its own normalisation conventions before tokenisation:
# - Replace all @mentions -> @USER
# - Replace all URLs -> HTTPURL
# - Do NOT lowercase — BERTweet is case-sensitive and was trained on raw tweet casing

def preprocess_tweet(text):
    text = re.sub(r'http\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    return text

# Apply preprocessing to all datasets
train_df['text'] = train_df['text'].apply(preprocess_tweet)
val_df['text'] = val_df['text'].apply(preprocess_tweet)
test_df['text'] = test_df['text'].apply(preprocess_tweet)

# Update Hugging Face Dataset objects with preprocessed text
train_dataset = Dataset.from_dict(train_df[['text', 'label']])
val_dataset = Dataset.from_dict(val_df[['text', 'label']])
test_dataset = Dataset.from_dict(test_df[['text', 'label']])

print("Preprocessing applied to tweet text in all datasets. Example:")
print("Original text (from train_df):", df.loc[train_df.index[0], 'text'])
print("Preprocessed text (from train_df):", train_df.loc[train_df.index[0], 'text'])


In [ ]:
#tokeniztion
!pip3 install emoji==0.6.0
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("vinai/bertweet-base", normalization=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

# Convert DataFrames to Hugging Face Dataset objects
from datasets import Dataset
train_dataset = Dataset.from_dict(train_df[['text', 'label']])
test_dataset = Dataset.from_dict(test_df[['text', 'label']])

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

In [ ]:
#train
!pip install -q evaluate
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# Load BERTweet for classification
model = AutoModelForSequenceClassification.from_pretrained("vinai/bertweet-base", num_labels=7)

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="macro")

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,        # From Section 3.7
    per_device_train_batch_size=16,
    num_train_epochs=5,         # From Section 3.7
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Start the training!
trainer.train()

In [ ]:
import os

# Define the path to save the model in Google Drive
MODEL_SAVE_PATH = "/content/drive/MyDrive/NLP_twitter/my_bertweet_model"

# Create the directory if it doesn't exist
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# Save the model and tokenizer
model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"Model and tokenizer saved to: {MODEL_SAVE_PATH}")

In [ ]:
#eval
# Evaluate on the test set
results = trainer.evaluate()
print(f"Final Macro F1 Score: {results['eval_f1']:.4f}")

# Test on a new tweet
def predict_tweet(text):
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    outputs = model(**inputs)
    probs = outputs.logits.softmax(dim=-1)
    pred_idx = probs.argmax().item()
    return candidate_labels[pred_idx]

print(predict_tweet("Don't sleep outside in the night time"))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Data Refinement Figure
labels = ['Raw Scraped', 'Deduplicated (Final)']
counts = [14457, 1255]

plt.figure(figsize=(8, 5))
sns.barplot(x=labels, y=counts, palette='viridis')
plt.title('Corpus Refinement: From Raw Scraping to Unique Data')
plt.ylabel('Number of Tweets')
plt.show()

# 2. Class Distribution Figure
# Replace with your actual: df['ai_label'].value_counts()
dist = df['ai_label'].value_counts()
plt.figure(figsize=(10, 6))
dist.plot(kind='barh', color='teal')
plt.title('Frequency of Malaria Discussion Categories')
plt.xlabel('Count')
plt.show()

In [ ]:
# This gets your public IP which you might need for the localtunnel password
import urllib
print("Password/Tunnel IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

# Start Streamlit and Tunnel
!streamlit run app.py & npx localtunnel --port 8501